# Tool Use Mini Demo - Local Weather Tool

This short notebook gives you a simple introduction to tool use before the main forensic case. In this warm-up, you will see the pattern in a familiar setting: ask a question, call a tool, inspect the result, and answer from evidence instead of guessing.


The image below shows the general Tool Use Pattern before we zoom into the local weather example.

<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f1a9fbda7-77a8-4a7a-ac2c-077fb98e53a6_716x552-1.gif" alt="General Tool Use Pattern" width="800"/>


## What You Will Do

This notebook is split into three parts so you can separate the idea of tool calling from the code that automates it, then try a similar question yourself.

**Part A: Understand the Tool Call**

1. Load the lab environment
2. Read the weather question and process diagram
3. Define the `get_weather` tool and inspect its signature
4. Watch the LLM choose `get_weather`
5. Run `get_weather` manually

**Part B: Let `ToolAgent` Run the Workflow**

6. See how `ToolAgent` invokes the tool
7. Let `ToolAgent` complete the full run

**Part C: Try a Similar Question**

8. Ask a new weather question about Phoenix on `2026-01-03`


## Part A: Understand the Tool Call

In Part A, slow down and look at the pieces one at a time. You will see the user question, the tool signature, the LLM's selected `<tool_call>`, and the weather result before the full agent workflow hides those details.


### Step 1: Set Up the Notebook

This cell does the basic setup work for the demo. It:

- reads this lab's `.env` file so the notebook knows which model to use
- connects to the model server
- adds the local `src/` folder so the notebook can import the course tool code

You do not need to memorize every line here. The main idea is that the notebook must load the model settings before it can run any tool demo. Open the notebook from the `lab2_tool_use_pattern` folder so this setup works as expected.


In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and builds the model client.
LAB_NAME = 'lab2_tool_use_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        'Expected .env in this folder. Copy .env.example to .env first.'
    )

load_dotenv(env_path, override=True)

MODEL = os.getenv("MODEL")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f"MODEL or OLLAMA_BASE_URL is missing from {env_path}")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


Repo root: /home/frank/projects/agentic-AI4-forensics
Lab folder: /home/frank/projects/agentic-AI4-forensics/lab2_tool_use_pattern


### Step 2: Read the Question and Process Diagram

Before you work on the vehicle case, start with one tiny familiar example. The warm-up question is:

`Should the detective bring an umbrella to a witness interview in Seattle on 2026-01-03?`

The goal is to see the tool-use loop in the simplest possible form:

1. ask a question the model should not guess
2. show the model which tools are available
3. let the model choose a tool call
4. run the tool to get outside information
5. use that tool output to answer from evidence

This warm-up uses a small local weather tool. It is only a teaching example, not part of the graded forensic report.


The diagram below shows the full tool-calling process for this weather example. From the student's perspective, `ToolAgent` manages the whole workflow: it sends the question and tool signature to the LLM, receives the LLM's proposed tool-call message, runs the actual Python tool, and returns the observation.

<img src="./figures/weather_tool_calling_process.svg" alt="Diagram showing how the LLM chooses get_weather from the question and tool signature, then ToolAgent validates and runs the tool" width="900"/>


### Step 3: Define the Tool and Inspect Its Signature

This cell defines one very small tool called `get_weather`. A tool is just a function the model is allowed to call when it needs outside information.

What to notice:

- the tool has a clear name: `get_weather`
- the tool expects specific inputs: `city` and `date`
- the tool returns structured information instead of a vague sentence
- the `@tool` wrapper creates a tool signature that the LLM can read
- after `@tool`, `get_weather` is a `Tool` object, so we use `get_weather.run(...)` to execute it

This is the key idea behind tool use: the model should ask the tool for information instead of guessing.


In [2]:
from agentic_patterns.tool_pattern.tool import tool, validate_arguments
from agentic_patterns.tool_pattern.tool_agent import TOOL_SYSTEM_PROMPT, ToolAgent
from agentic_patterns.utils.extraction import extract_tag_content

weather_lookup = {
    ("Seattle", "2026-01-03"): {
        "condition": "rain showers",
        "high_f": 46,
        "low_f": 39,
        "precipitation_chance": 0.85,
    },
    ("Phoenix", "2026-01-03"): {
        "condition": "sunny",
        "high_f": 68,
        "low_f": 47,
        "precipitation_chance": 0.00,
    },
    ("Boston", "2026-01-03"): {
        "condition": "cloudy",
        "high_f": 34,
        "low_f": 26,
        "precipitation_chance": 0.20,
    },
}

@tool
def get_weather(city: str, date: str):
    """
    Return a small local weather forecast from a mock lookup table.

    Args:
        city (str): City name such as "Seattle".
        date (str): Date in YYYY-MM-DD format.
    """
    normalized_city = city.strip().title()
    normalized_date = date.strip()
    record = weather_lookup.get((normalized_city, normalized_date))
    if record is None:
        return json.dumps(
            {"error": f"No weather record found for {normalized_city} on {normalized_date}"},
            indent=2,
        )
    return json.dumps(
        {"city": normalized_city, "date": normalized_date, **record},
        indent=2,
    )

warmup_question = "Should the detective bring an umbrella to a witness interview in Seattle on 2026-01-03?"

{
    "warmup_question": warmup_question,
    "tool_signature": json.loads(get_weather.fn_signature),
}


{'warmup_question': 'Should the detective bring an umbrella to a witness interview in Seattle on 2026-01-03?',
 'tool_signature': {'name': 'get_weather',
  'description': '\n    Return a small local weather forecast from a mock lookup table.\n\n    Args:\n        city (str): City name such as "Seattle".\n        date (str): Date in YYYY-MM-DD format.\n    ',
  'parameters': {'properties': {'city': {'type': 'str'},
    'date': {'type': 'str'}}}}}

### Step 4: How the Model Connects the Question to the Tool

Before any tool runs, the language model reads the user's question together with the available tool signature. In this step, we ask the LLM to propose the tool call, but we do not run the weather tool yet.

The full `TOOL_SYSTEM_PROMPT` lives in `src/agentic_patterns/tool_pattern/tool_agent.py`, but the important part says, in plain language:

```text
You are a function-calling AI model.
You are given function signatures inside <tools></tools> tags.
If a tool is needed, return a JSON object inside <tool_call></tool_call> tags.
The JSON must include: name, arguments, and id.
When you have enough information from tools, answer directly in plain text.
```

For this step, focus on the first part: the model chooses a `<tool_call>`. The plain-text final answer happens later, after a tool result comes back.

Do not mix up the two tag types:

| Tag | Who provides it? | What it means in plain language |
|---|---|---|
| `<tools></tools>` | The notebook or agent code gives this to the LLM | "Here are the tools you are allowed to use." |
| `<tool_call></tool_call>` | The LLM returns this when it chooses a tool | "I want to run this specific tool with these inputs." |

In short: `<tools>` lists the available options; `<tool_call>` is the model's selected option.

Example `<tools>` block shown to the LLM. In the real prompt, the tag and JSON are one continuous block; here we split them only so the JSON inside gets syntax colors:

```xml
<tools>
```

```json
{
  "name": "get_weather",
  "description": "Return a small local weather forecast",
  "parameters": {
    "properties": {
      "city": {"type": "str"},
      "date": {"type": "str"}
    }
  }
}
```

```xml
</tools>
```

If there are multiple tools, the `<tools>` block works like a menu with more than one entry. The model chooses the entry that best matches the user's question.

Example `<tool_call>` block returned by the LLM:

```xml
<tool_call>
```

```json
{
  "name": "get_weather",
  "arguments": {
    "city": "Seattle",
    "date": "2026-01-03"
  },
  "id": 1
}
```

```xml
</tool_call>
```

The tool signature is like a menu entry. It tells the model:

- the tool name: `get_weather`
- what the tool does: returns a local weather forecast
- what inputs the tool requires: `city` and `date`

Then the model compares the user question with that menu entry. In our question, the words `umbrella`, `Seattle`, and `2026-01-03` suggest that weather information is needed, so the model should choose `get_weather`.

How to read the `<tool_call>` object above:

- `name` means which tool the model chose
- `arguments` means the input values the tool needs
- `id` is just a label for this particular tool request

Important: the LLM is only choosing a tool-call message here. It is not running the Python function yet.

When you run the next cell, look for `"name": "get_weather"` and arguments for `Seattle` and `2026-01-03`. That is the moment where the LLM chooses the weather tool from the available tool signature.


In [3]:
# This cell shows how the LLM chooses a tool-call message before any tool runs.
# 1. The user's question contains clues: Seattle, date, umbrella/weather need.
# 2. The tool signature says there is a tool named get_weather that accepts city and date.
# 3. The LLM should respond with a <tool_call> that names get_weather and fills in those arguments.

model_matching_view = {
    "user_question": warmup_question,
    "available_tool_signature": json.loads(get_weather.fn_signature),
    "expected_tool_call": {
        "name": "get_weather",
        "arguments": {"city": "Seattle", "date": "2026-01-03"},
        "id": 1,
    },
}

tool_selection_prompt = f"""
Answer the user's question using the available tools if a tool is needed.

Question:
{warmup_question}

Return only the tool call. Do not answer the question yet.
""".strip()

tool_selection_messages = [
    {"role": "system", "content": TOOL_SYSTEM_PROMPT % get_weather.fn_signature},
    {"role": "user", "content": tool_selection_prompt},
]

llm_tool_selection = client.chat.completions.create(
    messages=tool_selection_messages,
    model=MODEL,
).choices[0].message.content

# The LLM response is text, so we search that text for content between
# <tool_call> and </tool_call>. The helper returns any matching block(s).
tool_call_blocks = extract_tag_content(str(llm_tool_selection), "tool_call")

# What `tool_call_blocks` usually looks like:
# tool_call_blocks.found   -> True
# tool_call_blocks.content -> ['{"name": "get_weather", "arguments": {...}, "id": 1}']

# Each extracted block should be JSON, so we parse it into a Python dictionary
# that is easier to inspect in the notebook output.
parsed_tool_calls = []
if tool_call_blocks.found:
    for block in tool_call_blocks.content:
        try:
            parsed_tool_calls.append(json.loads(block.strip()))
        except json.JSONDecodeError:
            parsed_tool_calls.append({"parse_error": "Could not parse tool call JSON", "raw_block": block})

{
    "what_the_model_saw": model_matching_view,
    "raw_llm_output": llm_tool_selection,
    "parsed_tool_calls": parsed_tool_calls,
    "expected_tool_name": "get_weather",
}


{'what_the_model_saw': {'user_question': 'Should the detective bring an umbrella to a witness interview in Seattle on 2026-01-03?',
  'available_tool_signature': {'name': 'get_weather',
   'description': '\n    Return a small local weather forecast from a mock lookup table.\n\n    Args:\n        city (str): City name such as "Seattle".\n        date (str): Date in YYYY-MM-DD format.\n    ',
   'parameters': {'properties': {'city': {'type': 'str'},
     'date': {'type': 'str'}}}},
  'expected_tool_call': {'name': 'get_weather',
   'arguments': {'city': 'Seattle', 'date': '2026-01-03'},
   'id': 1}},
 'raw_llm_output': '<tool_call>\n{"name": "get_weather","arguments": {"city": "Seattle","date": "2026-01-03"},"id": 1}\n</tool_call>',
 'parsed_tool_calls': [{'name': 'get_weather',
   'arguments': {'city': 'Seattle', 'date': '2026-01-03'},
   'id': 1}],
 'expected_tool_name': 'get_weather'}

### Step 5: Run `get_weather` Manually

This cell shows manual tool use. You run the tool directly with specific arguments and inspect the result.

This is helpful because it separates two jobs:

- the tool gathers the outside information
- you can later turn that information into a plain-language answer

In other words, you are running the same tool call that the LLM proposed in Step 4, but you are running it yourself so you can clearly see the evidence the answer should come from.


In [4]:
# Because `@tool` wrapped the function into a Tool object, we call `.run(...)` to execute it.
# Then we turn its JSON text output into a Python dictionary.
manual_weather_observation = json.loads(get_weather.run(city="Seattle", date="2026-01-03"))

# Show the observation returned by the tool.
manual_weather_observation


{'city': 'Seattle',
 'date': '2026-01-03',
 'condition': 'rain showers',
 'high_f': 46,
 'low_f': 39,
 'precipitation_chance': 0.85}

## Part B: Let `ToolAgent` Run the Workflow

In Part B, you stop running the pieces by hand. `ToolAgent` manages the workflow: it sends the question and tool signature to the LLM, reads the returned `<tool_call>`, validates the inputs, runs the Python tool, and uses the tool result to help produce the final answer.


### Step 6: See How `ToolAgent` Invokes the Tool

Now that you have seen the LLM propose a tool call and manually run the same tool, this cell shows the core function-call step inside `ToolAgent`. `ToolAgent` then:

- validates the tool-call arguments
- looks up the tool by name in `tools_dict`
- executes the wrapped function with `chosen_tool.run(**validated_tool_call["arguments"])`

This is the moment where the weather function actually gets invoked by the agent code.


In [5]:
weather_tool_agent = ToolAgent(tools=[get_weather], client=client, model=MODEL)

# This example uses the same kind of tool-call dictionary the model proposed in Step 4.
example_tool_call = {
    "name": "get_weather",
    "arguments": {"city": "Seattle", "date": "2026-01-03"},
    "id": 1,
}

# `ToolAgent` validates the arguments before it runs the tool.
validated_tool_call = validate_arguments(example_tool_call, json.loads(get_weather.fn_signature))

# Next it finds the requested tool by name and invokes it.
chosen_tool = weather_tool_agent.tools_dict[validated_tool_call["name"]]
tool_result = chosen_tool.run(**validated_tool_call["arguments"])

{
    "tool_call_from_model": example_tool_call,
    "validated_tool_call": validated_tool_call,
    "selected_tool": chosen_tool.name,
    "tool_result": json.loads(tool_result),
}


{'tool_call_from_model': {'name': 'get_weather',
  'arguments': {'city': 'Seattle', 'date': '2026-01-03'},
  'id': 1},
 'validated_tool_call': {'name': 'get_weather',
  'arguments': {'city': 'Seattle', 'date': '2026-01-03'},
  'id': 1},
 'selected_tool': 'get_weather',
 'tool_result': {'city': 'Seattle',
  'date': '2026-01-03',
  'condition': 'rain showers',
  'high_f': 46,
  'low_f': 39,
  'precipitation_chance': 0.85}}

### Step 7: Let `ToolAgent` Complete the Full Run

Now that you have seen the model-selection step, the manual tool run, and the exact invocation step, this cell shows the full end-to-end version. The model reads the prompt plus the tool signature, decides the tool call, `ToolAgent` runs it, and the model writes the final answer from the returned observation.

Compare the five views in this notebook:

- in Step 3, you defined the tool and saw its signature
- in Step 4, you watched the LLM propose the `get_weather` tool call from the question and tool signature
- in Step 5, you ran the tool yourself
- in Step 6, you saw the exact line where `ToolAgent` invokes the tool
- in Step 7, the agent handles the whole process for you


In [6]:
weather_tool_agent = ToolAgent(tools=[get_weather], client=client, model=MODEL)

weather_tool_agent_prompt = f"""
Answer the user's question using the available tools if a tool is needed.

Question:
{warmup_question}

Return a short answer that cites the tool result and does not guess beyond it.
""".strip()

weather_tool_agent_answer = weather_tool_agent.run(user_msg=weather_tool_agent_prompt)
print(weather_tool_agent_answer)



Using Tool: get_weather

Tool call dict: 
{'name': 'get_weather', 'arguments': {'city': 'Seattle', 'date': '2026-01-03'}, 'id': 1}

Tool result: 
{
  "city": "Seattle",
  "date": "2026-01-03",
  "condition": "rain showers",
  "high_f": 46,
  "low_f": 39,
  "precipitation_chance": 0.85
}
The detective should bring an umbrella. The weather forecast for Seattle on 2026-01-03 indicates an 85% chance of rain showers.


## Part C: Try a Similar Question

Now it is your turn to try the same pattern with a new city. The tool is still `get_weather`, but the user question changes from Seattle to Phoenix.

Before you run the code, make a quick prediction:

- Which tool should the LLM choose?
- What should the `city` argument be?
- What should the `date` argument be?
- Based on the weather result, should the detective bring an umbrella?


### Step 8: Practice With Phoenix

This cell asks a similar question about `Phoenix` on `2026-01-03`. Run it and compare the actual result with your prediction.


In [7]:
# Practice question: same pattern as Seattle, but with a new city.
practice_question = "Should the detective bring an umbrella to a witness interview in Phoenix on 2026-01-03?"

# Before running, students should expect the LLM to choose this tool call:
# {"name": "get_weather", "arguments": {"city": "Phoenix", "date": "2026-01-03"}, "id": 1}
practice_expected_tool_call = {
    "name": "get_weather",
    "arguments": {"city": "Phoenix", "date": "2026-01-03"},
    "id": 1,
}

practice_tool_agent = ToolAgent(tools=[get_weather], client=client, model=MODEL)

practice_tool_agent_prompt = f"""
Answer the user's question using the available tools if a tool is needed.

Question:
{practice_question}

Return a short answer that cites the tool result and does not guess beyond it.
""".strip()

practice_answer = practice_tool_agent.run(user_msg=practice_tool_agent_prompt)

{
    "practice_question": practice_question,
    "expected_tool_call": practice_expected_tool_call,
    "agent_answer": practice_answer,
}



Using Tool: get_weather

Tool call dict: 
{'name': 'get_weather', 'arguments': {'city': 'Phoenix', 'date': '2026-01-03'}, 'id': 1}

Tool result: 
{
  "city": "Phoenix",
  "date": "2026-01-03",
  "condition": "sunny",
  "high_f": 68,
  "low_f": 47,
  "precipitation_chance": 0.0
}


{'practice_question': 'Should the detective bring an umbrella to a witness interview in Phoenix on 2026-01-03?',
 'expected_tool_call': {'name': 'get_weather',
  'arguments': {'city': 'Phoenix', 'date': '2026-01-03'},
  'id': 1},
 'agent_answer': 'The detective does not need an umbrella for the interview in Phoenix on 2026-01-03, as the weather forecast shows 0% precipitation chance.'}

## Next Step

Before you move on, notice that every step in this notebook used the same weather tool pattern. The main difference is how much of the tool-calling work you did yourself versus how much `ToolAgent` handled for you. In Part C, you also saw that the same pattern can transfer to a new but similar question.

Now move to `02_case_overview.md`, then open `03b_lab_notebook.ipynb` to apply the same idea to the vehicle-sale case.
